# Summit-Sim: Complete E2E Workflow Demo

Demonstrates the full Summit-Sim workflow:
1. **Scenario Generation** - AI generates wilderness rescue scenario
2. **Simulation** - Student navigates scenario with human-in-the-loop
3. **Debrief** - Performance analysis and learning report

All phases are traced to MLflow for visibility.

## 1. Setup & Imports

In [1]:
import warnings

# Suppress autologging warnings
warnings.filterwarnings("ignore", message=".*autologging.*")
warnings.filterwarnings("ignore", message=".*LangChain.*")

from summit_sim.agents.debrief import generate_debrief
from summit_sim.agents.generator import generate_scenario
from summit_sim.graphs.simulation import create_simulation_graph
from summit_sim.schemas import HostConfig, generate_scenario_id
from summit_sim.settings import settings
from summit_sim.tracing import enable_tracing, simulation_session

# Initialize MLflow tracing
enable_tracing()

print(f"✓ MLflow: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key: {bool(settings.openrouter_api_key)}")

✓ MLflow: https://mlflow.bhamm-lab.com
✓ Experiment: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key: True


## 2. Phase 1: Scenario Generation

Generate a complete wilderness rescue scenario from minimal host configuration.

In [2]:
# Configuration
config = HostConfig(
    num_participants=3,
    activity_type="hiking",
    difficulty="med",
    class_id="demo-class-2024",
)

print(
    f"Generating: {config.activity_type} scenario for {config.num_participants} participants"
)
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)
scenario_id = generate_scenario_id()

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Scenario ID: {scenario_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

Generating: hiking scenario for 3 participants
------------------------------------------------------------
✓ Title: Ridge Line Rescue
✓ Setting: High-altitude, exposed ridge line, 4 miles from the trailhead. It is 5 PM, winds are picking up, and the temperature is 50°F (10°C) and falling rapidly.
✓ Patient: 35yo male hiker, fell 5 feet down a rocky slope. Presenting with severe pain and swelling in the right ankle, appears anxious.
✓ Turns: 4
✓ Scenario ID: scn-f165fae7

Learning Objectives:
  • Perform a structured head-to-toe patient assessment in a wilderness setting
  • Recognize signs of compensated shock and differentiate it from simple stress response
  • Apply appropriate wilderness interventions for shock and evacuation decision-making


Trace(trace_id=tr-125b5c5f4d79c8c47d555393b6f310c6)

### Display Scenario Structure

In [3]:
for turn in scenario.turns:
    marker = "(START)" if turn.is_starting_turn else ""
    print(f"\n{'=' * 60}")
    print(f"Turn {turn.turn_id} {marker}")
    print(f"{'=' * 60}")
    print(
        turn.narrative_text[:200] + "..."
        if len(turn.narrative_text) > 200
        else turn.narrative_text
    )
    print("\nChoices:")
    for choice in turn.choices:
        mark = "✓" if choice.is_correct else " "
        next_turn = (
            "END" if choice.next_turn_id is None else f"Turn {choice.next_turn_id}"
        )
        print(f"  [{mark}] {choice.description[:60]}... → {next_turn}")


Turn 0 (START)
You arrive at the scene of a fall on an exposed, breezy ridge. Your patient is a 35-year-old male sitting on a rock. He is holding his right ankle, which is visibly swollen. He appears anxious and is ...

Choices:
  [✓] Perform a thorough head-to-toe assessment before focusing on... → Turn 1
  [ ] Immediately splint the injured ankle to prepare for hiking o... → Turn 1

Turn 1 
Your head-to-toe check confirms a deformed right ankle. During your check of the core, he flinches when you palpate his left upper quadrant of the abdomen and describes a 'dull, deep ache' despite no ...

Choices:
  [✓] Recognize the abdominal pain, treat for potential shock, and... → Turn 2
  [ ] Focus exclusively on splinting the deformed ankle so you can... → Turn 2

Turn 2 
The patient has become pale, clammy, and says he feels slightly dizzy. His pulse is now rapid and thready. He is rapidly deteriorating. What is your action plan?

Choices:
  [✓] Lay him down, keep him warm, initiate shock 

## 3. Phase 2: Simulation

Run the interactive simulation with human-in-the-loop.
The graph will pause at each turn for you to make a choice.

In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Create graph with memory
memory = InMemorySaver()
graph = create_simulation_graph(checkpointer=memory)

# Initialize state
initial_state = {
    "scenario_draft": scenario,
    "current_turn_id": scenario.starting_turn_id,
    "transcript": [],
    "is_complete": False,
    "key_learning_moments": [],
    "last_selected_choice": None,
    "simulation_result": None,
    "scenario_id": scenario_id,
    "class_id": config.class_id,
}

print("✓ Graph initialized")
print(f"✓ Starting at turn {initial_state['current_turn_id']}")

✓ Graph initialized
✓ Starting at turn 0


In [5]:
# Run simulation with MLflow session tracking
from summit_sim.tracing import log_simulation_results

with simulation_session(config, scenario_id=scenario_id) as (session_id, graph_config):
    print(f"\n📋 Session ID: {session_id}")
    print("🎮 Starting simulation...\n")
    print("=" * 60)
    print("INTERACTIVE SIMULATION")
    print("=" * 60 + "\n")

    current_state = initial_state
    turn_count = 0
    simulation_complete = False

    while not simulation_complete:
        turn_count += 1

        # Only pass state on first turn
        input_state = initial_state if turn_count == 1 else None

        async for event in graph.astream(input_state, graph_config):
            if "__interrupt__" in event:
                # Handle human-in-the-loop
                interrupt_obj = event["__interrupt__"]
                if hasattr(interrupt_obj, "value"):
                    interrupt_data = interrupt_obj.value
                elif isinstance(interrupt_obj, (list, tuple)):
                    interrupt_data = (
                        interrupt_obj[0].value
                        if hasattr(interrupt_obj[0], "value")
                        else interrupt_obj[0]
                    )
                else:
                    interrupt_data = interrupt_obj

                print(f"\n--- TURN {turn_count} ---\n")
                print(interrupt_data["narrative"])
                print("\nChoices:")
                for i, choice in enumerate(interrupt_data["choices"], 1):
                    print(f"  {i}. {choice['description']}")

                # Get user input
                while True:
                    try:
                        sel = int(input("\nSelect choice (number): ")) - 1
                        if 0 <= sel < len(interrupt_data["choices"]):
                            break
                        print("Invalid selection")
                    except ValueError:
                        print("Please enter a number")

                selected = interrupt_data["choices"][sel]
                current_state = await graph.ainvoke(
                    Command(resume={"choice_id": selected["choice_id"]}), graph_config
                )
                break

            # Check completion
            if event == {}:
                simulation_complete = True
                break

        # Check if complete
        if current_state.get("is_complete"):
            simulation_complete = True

        # Safety limit
        if turn_count > 10:
            print("\nSafety limit reached")
            break

    print("\n" + "=" * 60)
    print("SIMULATION COMPLETE")
    print("=" * 60)

    # Log results
    log_simulation_results(
        transcript=current_state["transcript"],
        is_complete=current_state["is_complete"],
        key_learning_moments=current_state["key_learning_moments"],
    )
    print("\n✓ Results logged to MLflow")
    print(f"✓ Total turns: {len(current_state['transcript'])}")
    print(f"✓ Learning moments: {len(current_state['key_learning_moments'])}")

2026/03/23 21:21:04 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fb69b3af650> at 0x7fb65b5e7d80> was created in a different Context



📋 Session ID: 1d118f26-3258-4342-bd7a-1e85b04588d4
🎮 Starting simulation...

INTERACTIVE SIMULATION



2026/03/23 21:21:05 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fb69b3af650> at 0x7fb65b5eee00> was created in a different Context



--- TURN 1 ---

You arrive at the scene of a fall on an exposed, breezy ridge. Your patient is a 35-year-old male sitting on a rock. He is holding his right ankle, which is visibly swollen. He appears anxious and is complaining of pain. You are 4 miles from the trailhead and sunset is approaching. How do you proceed?

Choices:
  1. Perform a thorough head-to-toe assessment before focusing on the ankle.
  2. Immediately splint the injured ankle to prepare for hiking out as quickly as possible.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
2026/03/23 21:21:14 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fb69b3af650> at 0x7fb65b5853c0> was created in a different Context
2026/03/23 21:21:20 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fb69b3af650> at 0x7fb65b584ec0> was created in a different Context
2026/03/23 21:21:20 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fb69b3af650> at 0x7fb65b5776c0> was created in a different Context
2026/03/23 21:21:20 WARNING mlflow.utils.autologging_utils: Encountered un

...


--- TURN 4 ---

You have initiated shock treatment (warmth, horizontal position) and he seems slightly more alert, though he remains pale and tachycardic. Your satellite device shows you have cell service. How do you conclude your management? (Scenario ends - Decision made.)

Choices:
  1. Keep the patient warm and horizontal, communicate location to search and rescue, and wait for professional aid.
  2. Construct an makeshift litter and attempt to carry him out immediately to get him to the car.


2026/03/23 21:21:39 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NoneType' object has no attribute 'set_span_type'. For full traceback, set logging level to debug.
Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/23 21:21:39 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fb69b3af650> a


SIMULATION COMPLETE

✓ Results logged to MLflow
✓ Total turns: 4
✓ Learning moments: 8
🏃 View run sim-demo-class-2024-hiking-3p-med at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/32bc0abc094e4397b0e6d41ee32437b1
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


[Trace(trace_id=tr-89c8fa7a9034cf696bff9ed37efc23e3), Trace(trace_id=tr-3bfd061a8fe4c75b8b76d95d6aeddeda), Trace(trace_id=tr-26ad9c4e4b282ed777c97f8960ff092b), Trace(trace_id=tr-acef3f4d4f49f628375e1069322fcbaa)]

2026/03/23 21:21:46 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fb69b3af650> at 0x7fb65be08240> was created in a different Context


## 4. Phase 3: Debrief

Generate comprehensive performance report from simulation transcript.

In [6]:
print("\nGenerating debrief report...\n")

debrief_report = await generate_debrief(
    transcript=current_state["transcript"],
    scenario_draft=scenario,
    scenario_id=scenario_id,
)

print("=" * 60)
print("DEBRIEF REPORT")
print("=" * 60)

print(f"\n📊 Final Score: {debrief_report.final_score:.1f}%")
status_emoji = "✅" if debrief_report.completion_status == "pass" else "❌"
print(f"{status_emoji} Status: {debrief_report.completion_status.upper()}")

print("\n📝 Summary:")
print(debrief_report.summary)

if debrief_report.key_mistakes:
    print("\n⚠️ Key Mistakes:")
    for mistake in debrief_report.key_mistakes:
        print(f"  • {mistake}")

if debrief_report.strong_actions:
    print("\n💪 Strong Actions:")
    for action in debrief_report.strong_actions:
        print(f"  • {action}")

if debrief_report.teaching_points:
    print("\n📚 Teaching Points:")
    for point in debrief_report.teaching_points:
        print(f"  • {point}")

if debrief_report.best_next_actions:
    print("\n🎯 Recommendations:")
    for rec in debrief_report.best_next_actions:
        print(f"  • {rec}")


Generating debrief report...

DEBRIEF REPORT

📊 Final Score: 100.0%
✅ Status: PASS

📝 Summary:
You performed exceptionally well throughout this scenario. You demonstrated a high level of clinical maturity by avoiding tunnel vision, accurately identifying internal trauma, and prioritizing patient stability over the urge to immediately vacate the scene. Your performance reflects a strong grasp of both assessment protocols and the practical realities of wilderness first aid.

💪 Strong Actions:
  • Consistently utilized a systematic head-to-toe assessment to prevent cognitive bias.
  • Correctly identified abdominal pain as a priority over the visible ankle injury, avoiding the 'distraction injury' trap.
  • Successfully prioritized hemodynamic stability over mobility by initiating early shock management.
  • Demonstrated sound judgment in choosing to wait for professional extraction rather than attempting a hazardous self-evacuation.

📚 Teaching Points:
  • The importance of a 'distracti

Trace(trace_id=tr-d74eeab9f546d91ef0742fe9ba1e014a)

## 5. Results Summary

Complete transcript and learning moments from the simulation.

In [7]:
print("\n" + "=" * 60)
print("COMPLETE TRANSCRIPT")
print("=" * 60 + "\n")

for i, entry in enumerate(current_state["transcript"], 1):
    status = "✓" if entry["was_correct"] else "✗"
    print(f"Turn {i} {status}")
    print(f"  Situation: {entry['turn_narrative'][:80]}...")
    print(f"  Choice: {entry['choice_description']}")
    print(f"  Feedback: {entry['feedback'][:100]}...")
    print()


COMPLETE TRANSCRIPT

Turn 1 ✓
  Situation: You arrive at the scene of a fall on an exposed, breezy ridge. Your patient is a...
  Choice: Perform a thorough head-to-toe assessment before focusing on the ankle.
  Feedback: Excellent clinical judgment. By choosing a thorough head-to-toe assessment, you avoided the common p...

Turn 2 ✓
  Situation: Your head-to-toe check confirms a deformed right ankle. During your check of the...
  Choice: Recognize the abdominal pain, treat for potential shock, and splint the ankle carefully.
  Feedback: Excellent work. By identifying the left upper quadrant tenderness rather than focusing solely on the...

Turn 3 ✓
  Situation: The patient has become pale, clammy, and says he feels slightly dizzy. His pulse...
  Choice: Lay him down, keep him warm, initiate shock protocol, and stabilize for rescue/evacuation.
  Feedback: Excellent decision. By recognizing the subtle signs of shock, you correctly prioritized the patient'...

Turn 4 ✓
  Situation: You h

In [8]:
print("\n" + "=" * 60)
print("KEY LEARNING MOMENTS")
print("=" * 60 + "\n")

for i, moment in enumerate(current_state["key_learning_moments"], 1):
    print(f"{i}. {moment}")


KEY LEARNING MOMENTS

1. Always perform a comprehensive head-to-toe assessment before initiating treatment, even when a glaring injury is present.
2. Anxiety and restlessness are classic, often overlooked, early signs of compensated shock that can easily be mistaken for a normal reaction to pain.
3. Always perform a thorough head-to-toe assessment; external, obvious injuries (like an ankle sprain) often distract from more critical internal trauma.
4. In a remote setting, if you suspect internal injuries, you must treat for shock preemptively; don't wait for vital signs to deteriorate into decompensated shock.
5. Always look beyond the obvious injury to identify systemic red flags like pale/clammy skin, dizziness, and a rapid, thready pulse.
6. When systemic shock secondary to internal trauma is suspected, the definitive field treatment is immobilization, thermal regulation, and calling for higher-level resources rather than self-evacuation.
7. Internal injury leading to shock is a lif

---
## ✅ Complete Workflow Demo

All phases executed:
- ✓ Scenario generated with AI
- ✓ Simulation completed with human-in-the-loop
- ✓ Debrief report generated
- ✓ All traces logged to MLflow

View traces in MLflow UI at your configured tracking URI.